# Llama 3.2 1B Instruct: DPO Only

Recipe: start from `meta-llama/Llama-3.2-1B-Instruct` and apply DPO directly. The key control is `PROMPT_FORMAT = "chat_template"`: DPO uses the tokenizer's own Llama chat template, preserving the interface the instruction-tuned checkpoint already expects.

In [ ]:
from pathlib import Path
import os

# Kaggle: add HF_TOKEN under Add-ons > Secrets. Meta Llama models are gated,
# so your Hugging Face account must have access to the selected model.
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
except Exception as exc:
    print(f"No Kaggle HF_TOKEN secret loaded: {exc}")

if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"])

PROJECT_DIR = Path("/kaggle/working/qwen-dpo-alignment")
REPO_URL = "https://github.com/ahmad903501/qwen-dpo-alignment.git"
if not PROJECT_DIR.exists():
    !git clone {REPO_URL} {PROJECT_DIR}
%cd {PROJECT_DIR}
!git pull --ff-only
!python -m pip install -q -r requirements.txt
!python -m pip install -q -e . --no-deps

In [ ]:
from pathlib import Path

MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"
PROMPT_FORMAT = "chat_template"  # Critical: use the tokenizer's native Llama chat template.
DATASET_DPO = "HuggingFaceH4/ultrafeedback_binarized"

RUN_ROOT = Path("outputs/llama32_1b_instruct_dpo_only")
DPO_OUT = RUN_ROOT / "dpo"

DPO_TRAIN_EXAMPLES = 20000
DPO_VAL_EXAMPLES = 1000
MAX_LENGTH = 384
MAX_PROMPT_LENGTH = 192  # Kept for API consistency; chat template mode uses MAX_LENGTH for truncation.
SEED = 42
EVAL_STEPS = 64

## DPO

This run isolates the value of starting from the instruction-tuned checkpoint. It uses the same DPO dataset, beta, sample counts, max length, and accumulation settings as the base SFT-then-DPO path, but skips SFT and formats prompts with the model's native chat template.

In [ ]:
!python scripts/train_dpo.py \
  --model_name_or_path {MODEL_NAME} \
  --dataset_name {DATASET_DPO} \
  --train_split train \
  --val_split test \
  --output_dir {DPO_OUT} \
  --max_train_examples {DPO_TRAIN_EXAMPLES} \
  --max_val_examples {DPO_VAL_EXAMPLES} \
  --max_length {MAX_LENGTH} \
  --max_prompt_length {MAX_PROMPT_LENGTH} \
  --prompt_format {PROMPT_FORMAT} \
  --per_device_batch_size 1 \
  --gradient_accumulation_steps 32 \
  --num_epochs 1 \
  --learning_rate 5e-7 \
  --beta 0.1 \
  --logging_steps 5 \
  --eval_steps {EVAL_STEPS} \
  --seed {SEED}

## Training Curves

In [ ]:
!python scripts/plot_training_curves.py \
  --metrics {DPO_OUT}/metrics.jsonl \
  --output_dir {RUN_ROOT}/plots \
  --prefix llama32_1b_instruct_dpo

## Optional Pairwise Comparison

This generates Instruct-DPO outputs with the chat template and builds a randomized pairwise JSONL against the base SFT-then-DPO outputs when those are available.

In [ ]:
# Optional: compare against the base SFT->DPO checkpoint if that notebook's outputs are available
# in the same working directory or attached as a Kaggle input dataset.
EVAL_DIR = Path("outputs/llama32_1b_compare")
BASE_OUTPUTS = Path("outputs/llama32_1b_base_sft_then_dpo/alpaca_compare/llama32_1b_base_sft_dpo_outputs.json")
INSTRUCT_OUTPUTS = EVAL_DIR / "llama32_1b_instruct_dpo_outputs.json"

!python scripts/make_alpaca_subset.py --size 100 --seed {SEED} --output_dir {EVAL_DIR}
!python scripts/generate_alpaca.py \
  --model_name_or_path {DPO_OUT}/final \
  --subset_path {EVAL_DIR}/alpacaeval_100_seed{SEED}.json \
  --output_path {INSTRUCT_OUTPUTS} \
  --generator_name llama32_1b_instruct_dpo \
  --prompt_format chat_template \
  --batch_size 4 \
  --max_new_tokens 256 \
  --temperature 0.0

if BASE_OUTPUTS.exists():
    !python scripts/build_pairwise_eval.py \
      --a_outputs {BASE_OUTPUTS} \
      --b_outputs {INSTRUCT_OUTPUTS} \
      --a_name llama32_1b_base_sft_dpo \
      --b_name llama32_1b_instruct_dpo \
      --output_path {EVAL_DIR}/base_sft_dpo_vs_instruct_dpo_pairs.jsonl \
      --seed {SEED}
else:
    print(f"Base outputs not found at {BASE_OUTPUTS}. Run the base notebook generation cell or attach its outputs, then rerun this cell.")